In [1]:
%pip install pennylane pennylane-lightning qiskit qiskit-aer matplotlib scipy pyqsp --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pennylane as qml
import numpy as np

def create_messy_VH(raw_matrix):
    """
    Constructs a mathematically perfect, but highly chaotic (1, 2, 0)-block-encoding.
    """
    # 1. Standardize H (Hermitian and Scaled)
    H_hermitian = (raw_matrix + raw_matrix.conj().T) / 2.0
    norm = np.linalg.norm(H_hermitian, ord=2)
    H = H_hermitian / (norm + 0.1) 
    
    # 2. Get PennyLane's "Lazy" Block Encoding
    dev = qml.device("default.qubit", wires=3)
    @qml.qnode(dev)
    def lazy_unitary():
        qml.BlockEncode(H, wires=[0, 1, 2])
        return qml.state()
    U_lazy = qml.matrix(lazy_unitary)()
    
    # 3. Build the Junk-Mixing Matrix (W)
    # We want a 4x4 matrix for the 2 ancilla qubits.
    # State |00> remains untouched (1.0). The other 3 states are mixed using a 3x3 DFT.
    W_anc = np.zeros((4, 4), dtype=complex)
    W_anc[0, 0] = 1.0 
    
    # A standard 3x3 unitary matrix to mix the junk states symmetrically
    w = np.exp(2j * np.pi / 3)
    W_junk_mixer = np.array([[1, 1, 1],
                             [1, w, w**2],
                             [1, w**2, w]]) / np.sqrt(3)
    
    W_anc[1:, 1:] = W_junk_mixer # Apply the mixer ONLY to the junk states
    
    # Expand W to apply to the whole 3-qubit system (W_anc ⊗ I_data)
    W_full = np.kron(W_anc, np.eye(2))
    
    # 4. Create the Messy Unitary!
    V_messy = W_full @ U_lazy
    
    return H, V_messy

In [3]:
# --- Run and Verify ---
messy_target = np.array([[2.0 + 1j, 3.0], 
                         [-1.0, 4.0 - 2j]])

H, V_H = create_messy_VH(messy_target)

np.set_printoptions(
    linewidth=150,     
    precision=3,       
    suppress=True,     
    formatter={'complexfloat': lambda x: f"{x.real:.3f}" if np.abs(x.imag) < 1e-8 else f"{x.real:.3f}+{x.imag:.3f}j"}
)

print(H, "\n\n")

print(V_H)


[[0.443 0.222]
 [0.222 0.886]] 


[[0.393 0.196 0.884 -0.159 0.000 0.000 0.000 0.000]
 [0.196 0.785 -0.159 0.565 0.000 0.000 0.000 0.000]
 [0.511 -0.092 -0.227 -0.113 0.577 0.000 0.577 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 0.577 0.000 0.577]
 [0.511 -0.092 -0.227 -0.113 -0.289+0.500j 0.000 -0.289+-0.500j 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 -0.289+0.500j 0.000 -0.289+-0.500j]
 [0.511 -0.092 -0.227 -0.113 -0.289+-0.500j 0.000 -0.289+0.500j 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 -0.289+-0.500j 0.000 -0.289+0.500j]]


In [6]:
import pennylane as qml
import numpy as np

def build_lcu_encoding(V_matrix):
    """
    Constructs the spatial-efficient LCU with native 1/2 scaling.
    Uses asymmetric RY rotations on the Control qubit to mathematically 
    force the 1/2 factor without needing an extra ancilla.
    """
    wire_order = ["C", "A_0", "A_1", "S"]
    dev_efficient = qml.device("default.qubit", wires=wire_order)

    V_dagger = np.conj(V_matrix.T)
    R_mat = np.diag([-1, 1, 1, 1])

    @qml.qnode(dev_efficient)
    def scaled_lcu_circuit(V_mat, V_dag):
        # 1. ASYMMETRIC PREP: RY(5pi/6)
        # Prepares the state cos(75°)|0> + sin(75°)|1>
        qml.RY(5 * np.pi / 6, wires="C")
        
        # 2. SELECT (The Qubitization W Operator - completely unchanged)
        def apply_W():
            qml.QubitUnitary(V_mat, wires=["A_0", "A_1", "S"])
            qml.QubitUnitary(R_mat, wires=["A_0", "A_1"])
            qml.QubitUnitary(V_dag, wires=["A_0", "A_1", "S"])
            
        qml.ctrl(apply_W, control="C")()
        
        # 3. ASYMMETRIC PREP_dagger: RY(-pi/6)
        # Projects the amplitude onto the 15° axis
        qml.RY(-np.pi / 6, wires="C")
        
        return qml.state()

    U_LCU = qml.matrix(scaled_lcu_circuit, wire_order=wire_order)(V_matrix, V_dagger)
    
    # Extract the top-left block (Control=|0>, A_0=|0>, A_1=|0>)
    encoded_result = U_LCU[0:2, 0:2]
    
    return U_LCU, encoded_result

In [7]:
V_I_minus_H2_over_2, resulting_block = build_lcu_encoding(V_H)
H_encoded = V_H[0:2, 0:2]
math_target = (np.eye(2) - (H_encoded @ H_encoded)) / 2.0
print("H=")
print(H)
print("VH=")
print(V_H)
print("--- LCU Output ---")
print(resulting_block)
print("\n--- Math Target ---")
print(math_target)
print(f"\nMatch: {np.allclose(resulting_block, math_target)}")
print("\n--- Full LCU result ---")
print(V_I_minus_H2_over_2)

H=
[[0.443 0.222]
 [0.222 0.886]]
VH=
[[0.393 0.196 0.884 -0.159 0.000 0.000 0.000 0.000]
 [0.196 0.785 -0.159 0.565 0.000 0.000 0.000 0.000]
 [0.511 -0.092 -0.227 -0.113 0.577 0.000 0.577 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 0.577 0.000 0.577]
 [0.511 -0.092 -0.227 -0.113 -0.289+0.500j 0.000 -0.289+-0.500j 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 -0.289+0.500j 0.000 -0.289+-0.500j]
 [0.511 -0.092 -0.227 -0.113 -0.289+-0.500j 0.000 -0.289+0.500j 0.000]
 [-0.092 0.326 -0.113 -0.453 0.000 -0.289+-0.500j 0.000 -0.289+0.500j]]
--- LCU Output ---
[[0.404 -0.116]
 [-0.116 0.173]]

--- Math Target ---
[[0.404 -0.116]
 [-0.116 0.173]]

Match: True

--- Full LCU result ---
[[0.404 -0.116 -0.158 -0.024 -0.000 -0.000 -0.000 -0.000 -0.892 -0.031 -0.042 -0.006 -0.000 0.000 -0.000 0.000]
 [-0.116 0.173 -0.024 -0.206 -0.000 0.000 -0.000 -0.000 -0.031 -0.954 -0.006 -0.055 0.000 -0.000 0.000 -0.000]
 [-0.158 -0.024 0.096 0.116 -0.000 -0.000 0.000 0.000 -0.042 -0.006 -0.974 0.031 0.000 0.000 0.000